# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação
### Aula 1: Linhagem das CNNs, ResNet-34 do Zero e Transfer Learning no PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_01_cnn_architectures/aula_01_cnn_architectures.ipynb)

---

### 🎯 Objetivos de Aprendizagem
1. **Compreender o Paradoxo da Degradação e a Solução Residual**: Entender matematicamente por que redes muito profundas sofrem sem atalhos residuais ($F(x) + x$).
2. **Implementar a ResNet-34 do Zero em PyTorch**: Construir o `BasicBlock` com conexões residuais e montar os 4 estágios convolucionais modulares.
3. **Dominar o Transfer Learning Moderno**: Aplicar a nova API de *Weights Enum* (`torchvision.models.ResNet34_Weights`), extração de características e *fine-tuning*.
4. **Comparar Desempenho no Mundo Real**: Treinar do zero vs. usar pesos pré-treinados no dataset de imagens do mundo real **CIFAR-10** (10 categorias de objetos e animais).

## 1. Configuração do Ambiente e Verificação de GPU

Primeiro, vamos importar as bibliotecas essenciais e verificar se o Google Colab está com acelerador de GPU ativo (**T4 / V100 / A100**).

In [ ]:
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet34, ResNet34_Weights

# Fixar sementes para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# Configurar dispositivo (GPU CUDA ou CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Dispositivo de Execução: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("   ⚠️ Atenção: Nenhuma GPU detectada. No Colab, vá em: Ambiente de Execução -> Alterar tipo de ambiente -> GPU")

## 2. Carregamento e Preparação do Dataset de Imagens (CIFAR-10)

O **CIFAR-10** é composto por 60.000 imagens coloridas (RGB) divididas em 10 classes do mundo real: aviões, carros, pássaros, gatos, cervos, cachorros, sapos, cavalos, navios e caminhões.

Aplicaremos técnicas modernas de **Data Augmentation** no conjunto de treino (`RandomCrop` e `RandomHorizontalFlip`) e a normalização com a média e desvio padrão do dataset.

In [ ]:
# Nomes das 10 classes do CIFAR-10 em português
CLASS_NAMES = [
    'Avião', 'Automóvel', 'Pássaro', 'Gato', 'Cervo',
    'Cachorro', 'Sapo', 'Cavalo', 'Navio', 'Caminhão'
]

# Estatísticas de normalização do CIFAR-10
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

# Transformações para Treino (com Data Augmentation)
train_transforms_scratch = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Transformações para Teste (apenas normalização determinística)
test_transforms_scratch = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Download dos datasets (executado diretamente na nuvem pelo Colab)
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms_scratch)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transforms_scratch)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Conjunto de Treino: {len(train_dataset)} imagens")
print(f"✅ Conjunto de Teste:  {len(test_dataset)} imagens")
print(f"✅ Número de Classes:   {len(CLASS_NAMES)}")

### 🖼️ Visualizando Amostras do Dataset com Desnormalização

In [ ]:
def unnormalize(tensor, mean=CIFAR_MEAN, std=CIFAR_STD):
    """Reverte a normalização para exibição correta das cores com Matplotlib."""
    img = tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(std) + np.array(mean)
    return np.clip(img, 0, 1)

# Pegar um lote de imagens
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for idx, ax in enumerate(axes.flat):
    ax.imshow(unnormalize(images[idx]))
    ax.set_title(f"{CLASS_NAMES[labels[idx]]}", fontsize=11, fontweight='bold', color='#0A345D')
    ax.axis('off')

plt.suptitle('Amostras do CIFAR-10 com Data Augmentation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Implementando a ResNet-34 do Zero em PyTorch

A ResNet (*Residual Network*, He et al., 2015) resolve o paradoxo da degradação adicionando uma **conexão de atalho (*skip connection*)**:

$$\mathbf{y} = \mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x}$$

A estrutura modular da ResNet-34 consiste em:
1. **`BasicBlock`**: Dois blocos de Convolução 3x3 com `BatchNorm2d` e ativação `ReLU`. Quando as dimensões espaciais diminuem ($stride=2$) ou os canais dobram, o atalho passa por uma convolução 1x1 com `stride=2` para compatibilizar as dimensões.
2. **4 Estágios Convolucionais**: Contendo `[3, 4, 6, 3]` blocos residuais respectivamente (com 64, 128, 256 e 512 canais).
3. **Cabeça de Classificação Global**: `AdaptiveAvgPool2d((1, 1))` seguido por uma camada linear `nn.Linear(512, num_classes)`.

In [ ]:
class BasicBlock(nn.Module):
    """
    Bloco Construtivo Residual Fundamental da ResNet-18/34.
    Executa: Conv 3x3 -> BN -> ReLU -> Conv 3x3 -> BN -> Soma Residual (+ x) -> ReLU.
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        
        # Primeira Convolução 3x3 (aplica stride=2 se houver downsampling)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        
        # Segunda Convolução 3x3 (stride fixo em 1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        
        # Conexão de Atalho (Shortcut)
        # Se o stride > 1 ou os canais mudarem, ajustamos a dimensão de x com conv 1x1
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # Aprendizado Residual: somamos o sinal original de entrada identity antes da ReLU
        out += self.shortcut(identity)
        out = self.relu(out)
        return out


class ResNet34FromScratch(nn.Module):
    """
    Arquitetura Completa da ResNet-34 implementada do zero em PyTorch.
    """
    def __init__(self, num_classes=10):
        super(ResNet34FromScratch, self).__init__()
        self.in_channels = 64
        
        # Camada Inicial (Stem): Conv 3x3 + BN + ReLU
        # Para imagens CIFAR (32x32), usamos kernel 3x3 stride 1 para preservar resolução inicial
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)
        self.relu  = nn.ReLU(inplace=True)
        
        # 4 Estágios Residuais (Layers com 3, 4, 6 e 3 blocos)
        self.layer1 = self._make_layer(BasicBlock, 64,  num_blocks=3, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, num_blocks=4, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, num_blocks=6, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, num_blocks=3, stride=2)
        
        # Cabeça de Classificação Global
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc       = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avg_pool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

# Instanciar e testar o forward pass
model_scratch = ResNet34FromScratch(num_classes=10).to(device)
dummy_input = torch.randn(2, 3, 32, 32).to(device)
dummy_output = model_scratch(dummy_input)

total_params = sum(p.numel() for p in model_scratch.parameters() if p.requires_grad)
print(f"✅ ResNet-34 Instanciada com Sucesso!")
print(f"   Tensor de Entrada: {dummy_input.shape}")
print(f"   Tensor de Saída:   {dummy_output.shape}")
print(f"   Total de Parâmetros Treináveis: {total_params / 1e6:.2f} Milhões")

## 4. Loop de Treinamento e Avaliação Modular

Vamos criar funções de treino e validação com rastreamento de perda (`CrossEntropyLoss`), cálculo de acurácia Top-1 e agendador de taxa de aprendizado (*Cosine Annealing*).

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Treina o modelo por uma única época."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100.0
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """Avalia o modelo no conjunto de teste sem calcular gradientes."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    val_loss = running_loss / total
    val_acc = (correct / total) * 100.0
    return val_loss, val_acc

### 🚀 Treinando a ResNet-34 do Zero (Scratch Training)

In [ ]:
EPOCHS = 5
criterion = nn.CrossEntropyLoss()
optimizer_scratch = optim.AdamW(model_scratch.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_scratch = optim.lr_scheduler.CosineAnnealingLR(optimizer_scratch, T_max=EPOCHS)

history_scratch = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🏋️ Iniciando Treinamento da ResNet-34 (DO ZERO)...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc = train_one_epoch(model_scratch, train_loader, criterion, optimizer_scratch, device)
    v_loss, v_acc = evaluate(model_scratch, test_loader, criterion, device)
    scheduler_scratch.step()
    
    history_scratch['train_loss'].append(t_loss)
    history_scratch['train_acc'].append(t_acc)
    history_scratch['val_loss'].append(v_loss)
    history_scratch['val_acc'].append(v_acc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2f}% | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2f}%")

elapsed = time.time() - start_time
print(f"⏱️ Tempo total de treinamento (Do Zero): {elapsed:.1f} segundos")

## 5. Transfer Learning com ResNet-34 Pré-Treinada no ImageNet

Agora aplicaremos **Transfer Learning** utilizando a API moderna do TorchVision (**Weights Enum**):

```python
from torchvision.models import resnet34, ResNet34_Weights
weights = ResNet34_Weights.DEFAULT
model = resnet34(weights=weights)
```

### 🔑 O Fluxo do Transfer Learning:
1. **Carregar o Modelo com Pesos Oficiais**: `resnet34(weights=ResNet34_Weights.DEFAULT)`.
2. **Aproveitar o Pipeline de Transformações Automáticas**: `weights.transforms()` garante que as imagens recebam o mesmo redimensionamento (224x224) e normalização do ImageNet.
3. **Congelar o Backbone (Feature Extraction)**: `for p in model.parameters(): p.requires_grad = False`.
4. **Substituir a Camada Final**: Trocar `model.fc` por uma nova camada `nn.Linear(512, 10)`.

In [ ]:
# 1. Obter os pesos pré-treinados e as transformações oficiais
pretrained_weights = ResNet34_Weights.DEFAULT
pretrained_transforms = pretrained_weights.transforms()
print(f"🔍 Pipeline de Transformações Automáticas do TorchVision:")
print(pretrained_transforms)

# 2. Reconfigurar os DataLoaders com as transformações da ResNet-34 oficial
train_dataset_tl = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=pretrained_transforms)
test_dataset_tl  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=pretrained_transforms)

train_loader_tl = DataLoader(train_dataset_tl, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader_tl  = DataLoader(test_dataset_tl, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 3. Instanciar a ResNet-34 com os pesos pré-treinados
model_tl = resnet34(weights=pretrained_weights)

# 4. Feature Extraction: Congelar todas as camadas convolucionais pré-treinadas
for param in model_tl.parameters():
    param.requires_grad = False

# 5. Substituir a cabeça de classificação para 10 classes
in_features = model_tl.fc.in_features
model_tl.fc = nn.Linear(in_features, len(CLASS_NAMES))
model_tl = model_tl.to(device)

trainable_params_tl = sum(p.numel() for p in model_tl.parameters() if p.requires_grad)
total_params_tl     = sum(p.numel() for p in model_tl.parameters())
print(f"\n✅ Modelo Pré-Treinado Configurado:")
print(f"   Parâmetros Totais: {total_params_tl / 1e6:.2f} M")
print(f"   Parâmetros Treináveis (Apenas Cabeça FC): {trainable_params_tl / 1e3:.2f} mil pesos (apenas {trainable_params_tl/total_params_tl*100:.2f}% do total!)")

### ⚡ Treinando o Modelo com Transfer Learning (Feature Extraction)

In [ ]:
optimizer_tl = optim.AdamW(model_tl.fc.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_tl = optim.lr_scheduler.CosineAnnealingLR(optimizer_tl, T_max=EPOCHS)

history_tl = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("⚡ Iniciando Treinamento com Transfer Learning (Pesos ImageNet)...")
start_time_tl = time.time()

for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc = train_one_epoch(model_tl, train_loader_tl, criterion, optimizer_tl, device)
    v_loss, v_acc = evaluate(model_tl, test_loader_tl, criterion, device)
    scheduler_tl.step()
    
    history_tl['train_loss'].append(t_loss)
    history_tl['train_acc'].append(t_acc)
    history_tl['val_loss'].append(v_loss)
    history_tl['val_acc'].append(v_acc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2f}% | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2f}%")

elapsed_tl = time.time() - start_time_tl
print(f"⏱️ Tempo total de treinamento (Transfer Learning): {elapsed_tl:.1f} segundos")

## 6. Comparativo de Desempenho: Do Zero vs Transfer Learning

Vamos plotar as curvas de aprendizado lado a lado para demonstrar a superioridade do Transfer Learning em termos de velocidade de convergência e acurácia final.

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 1. Gráfico de Acurácia de Validação
ax1.plot(epochs_range, history_scratch['val_acc'], 'o--', color='#FF7043', label='ResNet-34 (Do Zero)', linewidth=2)
ax1.plot(epochs_range, history_tl['val_acc'], 's-', color='#1BB5D8', label='ResNet-34 (Transfer Learning)', linewidth=2.5)
ax1.set_title('Acurácia de Validação (Top-1 %)', fontsize=13, fontweight='bold', color='#0A345D')
ax1.set_xlabel('Época')
ax1.set_ylabel('Acurácia (%)')
ax1.set_ylim(40, 100)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='lower right', frameon=True)

# 2. Gráfico de Perda de Validação (Loss)
ax2.plot(epochs_range, history_scratch['val_loss'], 'o--', color='#FF7043', label='ResNet-34 (Do Zero)', linewidth=2)
ax2.plot(epochs_range, history_tl['val_loss'], 's-', color='#1BB5D8', label='ResNet-34 (Transfer Learning)', linewidth=2.5)
ax2.set_title('Perda de Validação (Cross-Entropy Loss)', fontsize=13, fontweight='bold', color='#0A345D')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

print("="*70)
print(f"📊 RESUMO COMPARATIVO APÓS {EPOCHS} ÉPOCAS:")
print(f"   • ResNet-34 (Do Zero):            Acurácia Validação = {history_scratch['val_acc'][-1]:.2f}%")
print(f"   • ResNet-34 (Transfer Learning):   Acurácia Validação = {history_tl['val_acc'][-1]:.2f}%")
print(f"   • Ganho Absoluto de Precisão:     +{history_tl['val_acc'][-1] - history_scratch['val_acc'][-1]:.2f}% com 99% menos parâmetros treinados!")
print("="*70)

## 7. Inferência Visual e Distribuição de Probabilidades Softmax

Vamos realizar a inferência em imagens de teste e plotar a imagem com a classe prevista, classe real e barra de probabilidades geradas pelo modelo pré-treinado.

In [ ]:
model_tl.eval()
test_images, test_labels = next(iter(test_loader_tl))
test_images = test_images.to(device)

with torch.no_grad():
    logits = model_tl(test_images)
    probs = torch.softmax(logits, dim=1)

# Exibir 4 predições com barras de confiança
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, ax in enumerate(axes.flat):
    img_display = test_images[i].cpu().numpy().transpose(1, 2, 0)
    # Desnormalizar com as estatísticas do ImageNet
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_display = np.clip(img_display * std + mean, 0, 1)
    
    top3_prob, top3_idx = torch.topk(probs[i], 3)
    top3_prob = top3_prob.cpu().numpy() * 100
    top3_classes = [CLASS_NAMES[idx] for idx in top3_idx.cpu().numpy()]
    
    true_class = CLASS_NAMES[test_labels[i]]
    pred_class = top3_classes[0]
    is_correct = (pred_class == true_class)
    
    ax.imshow(img_display)
    title_color = '#15803D' if is_correct else '#DC2626'
    ax.set_title(f"Real: {true_class} | Predito: {pred_class} ({top3_prob[0]:.1f}%)", color=title_color, fontweight='bold', fontsize=11)
    ax.axis('off')

plt.suptitle('Inferência Visual com Modelo ResNet-34 Pré-Treinado', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Desafios e Atividades Práticas Propostas

Para aprofundar seus conhecimentos, realize as seguintes experiências práticas no Google Colab:

1. **Fine-Tuning com Taxas de Aprendizado Diferenciais**:
   - Descongele as camadas `layer4` da ResNet-34 e configure o otimizador com taxa menor para o backbone e maior para a cabeça:
   ```python
   for param in model_tl.layer4.parameters():
       param.requires_grad = True
       
   optimizer = optim.AdamW([
       {'params': model_tl.layer4.parameters(), 'lr': 1e-5},
       {'params': model_tl.fc.parameters(),     'lr': 1e-3}
   ])
   ```
2. **Testar Outras Arquiteturas da Tabela 12-3**:
   - Substitua a ResNet-34 por `efficientnet_b0` ou `convnext_tiny` e compare a quantidade de parâmetros e a acurácia obtida no CIFAR-10.
3. **Medição de Memória VRAM**:
   - Use `torch.cuda.memory_allocated() / (1024**2)` para monitorar a pegada de memória durante o treinamento e durante a inferência com `torch.no_grad()`.